# 03 · Análisis del gasto público

**Objetivo:** responder las preguntas descriptivas del proyecto con el modelo gold:
¿cuánto se adjudica, cuándo, en qué, mediante qué procedimientos, quién compra y quién vende?

**Entradas:** tablas de `data/curated` (las mismas que usa Power BI).

In [ ]:
# Configuración del entorno (ejecutar primero)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# Raíz del proyecto: la carpeta actual o la primera superior con pyproject.toml.
CWD = Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents) if (p / "pyproject.toml").is_file()), CWD)
sys.path.insert(0, str(ROOT / "src"))  # permite importar el paquete aunque no esté instalado

CURATED = ROOT / "data" / "curated"      # capa gold
PROCESSED = ROOT / "data" / "processed"  # capa silver
if not (CURATED / "fact_proceso.parquet").exists():
    raise FileNotFoundError(
        "No hay datos procesados. Ejecuta primero en la terminal: contrataciones run --source sample"
    )

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
PALETA = ["#1F2A44", "#3E6A8A", "#7FA7C2", "#C9A227", "#B5533C", "#5B8C5A", "#8C8C8C"]
plt.rcParams.update({
    "figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.prop_cycle": plt.cycler(color=PALETA),
    "axes.titleweight": "bold", "axes.titlesize": 12,
})
millones = mticker.FuncFormatter(lambda x, _: f"{x / 1e6:,.0f} M")
print("Proyecto:", ROOT)

In [ ]:
def cargar(tabla):
    return pd.read_parquet(CURATED / f"{tabla}.parquet")

fp, fa = cargar("fact_proceso"), cargar("fact_adjudicacion")
entidades, proveedores = cargar("dim_entidad"), cargar("dim_proveedor")
metodos, categorias, fechas = cargar("dim_metodo"), cargar("dim_categoria"), cargar("dim_fecha")

# Vista plana equivalente a sql/views/vw_procesos.sql
procesos = (
    fp.merge(fechas[["fecha_key", "fecha", "anio", "anio_mes"]],
             left_on="fecha_convocatoria_key", right_on="fecha_key", how="left")
      .merge(metodos[["metodo_key", "metodo_detalle", "es_competitivo"]], on="metodo_key", how="left")
      .merge(categorias[["categoria_key", "categoria"]], on="categoria_key", how="left")
      .merge(entidades[["entidad_key", "nombre", "departamento"]], on="entidad_key", how="left")
      .rename(columns={"nombre": "entidad"})
)
print(f"{len(procesos):,} procesos | monto adjudicado S/ {procesos['monto_adjudicado_pen'].sum():,.0f}")

## 1. Evolución mensual del monto adjudicado

In [ ]:
mensual = procesos.groupby("anio_mes")["monto_adjudicado_pen"].sum()
ax = mensual.plot(marker="o", title="Monto adjudicado por mes de convocatoria (S/)")
ax.yaxis.set_major_formatter(millones); ax.set_xlabel(""); plt.xticks(rotation=45); plt.show()

## 2. ¿En qué se gasta? Categoría y procedimiento

In [ ]:
por_categoria = procesos.groupby("categoria")["monto_adjudicado_pen"].sum().sort_values()
ax = por_categoria.plot.barh(title="Monto adjudicado por categoría (S/)", color=PALETA[0])
ax.xaxis.set_major_formatter(millones); ax.set_ylabel(""); plt.show()

In [ ]:
por_metodo = procesos.groupby("metodo_detalle").agg(
    procesos=("ocid", "count"),
    monto=("monto_adjudicado_pen", "sum"),
    postores_promedio=("n_postores", "mean"),
).sort_values("monto", ascending=False)
por_metodo["% del monto"] = por_metodo["monto"] / por_metodo["monto"].sum()
por_metodo.style.format({"monto": "{:,.0f}", "postores_promedio": "{:.2f}", "% del monto": "{:.1%}"})

## 3. ¿Quién compra? Top 10 entidades

In [ ]:
top_entidades = (procesos.groupby(["entidad", "departamento"], dropna=False)
                 .agg(procesos=("ocid", "count"), monto=("monto_adjudicado_pen", "sum"))
                 .sort_values("monto", ascending=False).head(10))
top_entidades.style.format({"monto": "{:,.0f}"})

In [ ]:
por_departamento = procesos.groupby("departamento")["monto_adjudicado_pen"].sum().sort_values().tail(12)
ax = por_departamento.plot.barh(title="Monto adjudicado por departamento de la entidad (S/)",
                                color=PALETA[1])
ax.xaxis.set_major_formatter(millones); ax.set_ylabel(""); plt.show()

## 4. ¿Quién vende? Top 10 proveedores

In [ ]:
validas = fa[fa["es_valida"]].merge(proveedores, on="proveedor_key", how="left")
# Se agrupa por la llave del proveedor: los consorcios no tienen RUC propio.
top_proveedores = (validas.groupby("proveedor_key")
                   .agg(proveedor=("nombre", "first"), ruc=("ruc", "first"),
                        tipo=("tipo_persona", "first"), procesos_ganados=("ocid", "nunique"),
                        entidades=("entidad_key", "nunique"), monto=("monto_asignado_pen", "sum"))
                   .sort_values("monto", ascending=False).head(10).set_index("proveedor"))
top_proveedores.style.format({"monto": "{:,.0f}"}, na_rep="-")

In [ ]:
# Participación en el monto adjudicado por tipo de proveedor (incluye consorcios)
por_tipo = validas.groupby("tipo_persona", dropna=False)["monto_asignado_pen"].sum()
(por_tipo / por_tipo.sum()).sort_values(ascending=False).map("{:.1%}".format)

## 5. Distribuciones: montos y competencia

In [ ]:
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
valores = procesos["valor_referencial_pen"].dropna()
ax1.hist(np.log10(valores[valores > 0]), bins=40, color=PALETA[0])
ax1.set_title("Valor referencial (escala log10)"); ax1.set_xlabel("log10(S/)")
procesos["n_postores"].dropna().astype(int).clip(upper=10).value_counts().sort_index().plot.bar(
    ax=ax2, color=PALETA[2], title="Número de postores por proceso (10 = 10 o más)")
ax2.set_xlabel("postores"); plt.tight_layout(); plt.show()

## Conclusiones

Resume aquí los hallazgos principales (con datos reales): estacionalidad del gasto, concentración
por categoría y procedimiento, entidades y proveedores con mayor peso, y nivel de competencia.
Siguiente notebook: **04 · Indicadores de riesgo**.